In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


In [2]:
!pip install --pre pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.5/544.5 kB 810.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.0/307.0 kB 22.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... - \ | / - \ | / - \ | / - \ | / - done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Importing Libraries

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from pycaret.classification import *

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

## 2. Data Description

In this competition your task is to predict whether a passenger was transported to an alternate dimension during the Spaceship Titanic's collision with the spacetime anomaly. To help you make these predictions, you're given a set of personal records recovered from the ship's damaged computer system.

## 2.1. File and Data Field Descriptions


1. train.csv - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
    1. PassengerId - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
    1. HomePlanet - The planet the passenger departed from, typically their planet of permanent residence.
    1. CryoSleep - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
    1. Cabin - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
    1. Destination - The planet the passenger will be debarking to.
    1. Age - The age of the passenger.
    1. VIP - Whether the passenger has paid for special VIP service during the voyage.
    1. RoomService, FoodCourt, ShoppingMall, Spa, VRDeck - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
    1. Name - The first and last names of the passenger.
    1.Transported - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.
    
1. test.csv - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of
    Transported for the passengers in this set.
1. sample_submission.csv - A submission file in the correct format.
    1. PassengerId - Id for each passenger in the test set.
    1. Transported - The target. For each passenger, predict either True or False.

## 3. Let's Load the Data

In [4]:
train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
sub_sample = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')

train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Target Variable

We can see from the above output that our target variable is Transported and it was mentioned in Competition Overview too. It is a binary class variable, the answer is either True or False, which can be converted to 1 or 0.

## Predictor variables

We can also number of different predictor variables, among which some are categorical variables like Cryosleep, HomePlanet and VIP, and somer others are numerical variables like RoomService, FoodCourst, ShoppingMall etc, etc.

In [5]:
# Let's check dimensions of train and test set
print(f" Training set dimensions : {train.shape}")
print(f" Testing set dimensions : {test.shape}")

 Training set dimensions : (8693, 14)
 Testing set dimensions : (4277, 13)


## 4. EDA (Exploratory Data Analysis)

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [7]:
# Let's check NA'S values in each columns
train.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [8]:
train.corr()['Transported'].sort_values(ascending= False)

Transported     1.000000
FoodCourt       0.046566
ShoppingMall    0.010141
Age            -0.075026
VRDeck         -0.207075
Spa            -0.221131
RoomService    -0.244611
Name: Transported, dtype: float64

## 5. Feature Engineering

In [9]:
# Have to create copies of training and testing set, so we can have acces to original set if needed.
train['Transported'] = train['Transported'].astype(int)
train_copy = train.copy()
test_copy = test.copy()

Defining Neccessary funcions below to handle with missing values, converting categorical binary variables into numerical binary variables for purpose of training, splitting Name into First name and Last name and etc, etc.

In [10]:
# defining function for filling missing values of categorical variables

def fill_cat_var(df : pd.DataFrame):
    features = list(train_copy.select_dtypes(exclude = ['int64', 'float64']).columns)
    # removing PassengerId is it doesn't contain any na's value and has no use in training as well
    features.remove('PassengerId')
    for feat in features:
        df[feat].fillna(df[feat].mode()[0], inplace = True)
    return df

# defining function for filling missing values of numerical variables

def fill_num_var(df: pd.DataFrame):
    features = list(train_copy.select_dtypes(include = ['int64', 'float64']).columns)
    features.remove('Transported')
    for feat in features:
        df[feat].fillna(df[feat].median(), inplace = True)
    return df

# defining a function to convert binary to numerical 
def convert_binary_to_num(df: pd.DataFrame):
    features = ['VIP', 'CryoSleep']
    for feat in features:
        df[feat] = df[feat].astype(int)
    return df

# defining function to take linearlog transfromation of some of predictor variables

def linalg_transform(df : pd.DataFrame):
    features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for feat in features:
        df[feat] = np.log(df[feat])
    return df

# defining a function to split some composite column into different columns like Cabin and Name

def split_columns(df: pd.DataFrame):
    df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Num'] = df['Cabin'].apply(lambda x: x.split('/')[1])
    df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])
    
    df['LastName'] = df.Name.str.split(' ').str[1]
    df['Group'] = df['PassengerId'].apply(lambda x: x[0:4])
    df['Relative_group'] = df.groupby(['Group'])['LastName'].transform('count')
    return df

# defining a function to drop few columns that we don't need

def drop_coloumns(df: pd.DataFrame):
    df.drop(['Name', 'LastName', 'Group'], axis = 1, inplace = True)
    return df

Now, we will use pandas .pipe() function to apply all the above functions on our training and testing set sequentially as below.

In [11]:
%%time

train_copy = (train_copy.pipe(fill_cat_var).pipe(fill_num_var).pipe(convert_binary_to_num))
test_copy = (test_copy.pipe(fill_cat_var).pipe(fill_num_var).pipe(convert_binary_to_num))

train_copy = (train_copy.pipe(split_columns).pipe(drop_coloumns))
test_copy = (test_copy.pipe(split_columns).pipe(drop_coloumns))

CPU times: user 104 ms, sys: 6.69 ms, total: 111 ms
Wall time: 114 ms


In [12]:
# let's check train_copy with head function after transformation
train_copy.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side,Relative_group
0,0001_01,Europa,0,B/0/P,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,0.0,0.0,0,B,0,P,1
1,0002_01,Earth,0,F/0/S,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,549.0,44.0,1,F,0,S,1
2,0003_01,Europa,0,A/0/S,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,A,0,S,2
3,0003_02,Europa,0,A/0/S,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,A,0,S,2
4,0004_01,Earth,0,F/1/S,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,565.0,2.0,1,F,1,S,1


## 5.1. LabelEncoder

In [13]:
categorical_variables = list(train_copy.select_dtypes(exclude = ['int64', 'float64']).columns)
categorical_variables.remove('PassengerId')

for var in categorical_variables:
    print(var)
    label_encod = LabelEncoder()
    arr = np.concatenate((train_copy[var], test_copy[var])).astype(str)
    label_encod.fit(arr)
    train_copy[var] = label_encod.transform(train_copy[var].astype(str))
    test_copy[var] = label_encod.transform(test_copy[var].astype(str))

HomePlanet
Cabin
Destination
Deck
Num
Side


## 5.2. Standard Scalarization

Now, using standard scaler library from Scikit Learn to apply feature scalling to both train and test set.

In [14]:
# storing predictor variables in X and then do feature scalling

# train_copy.drop('PassengerId', axis = 1, inplace = True)
# test_copy.drop('PassengerId', axis = 1, inplace = True)


# features = list(train_copy.columns)
# scaler = StandardScaler()
# train_copy[features] = scaler.fit_transform(train_copy[features])
# test_copy[features] = scaler.fit_transform(test_copy[features])

## 6. Building Models

In [15]:
setup(data = train_copy, target = 'Transported',train_size = 0.95,
          fold_strategy = 'stratifiedkfold',
          fold = 5,
          fold_shuffle = True,
          remove_multicollinearity = True,
          normalize = True,
          normalize_method = 'robust')

,Description,Value
0,Session id,8278
1,Target,Transported
2,Target type,Binary
3,Original data shape,"(8693, 17)"
4,Transformed data shape,"(8693, 15)"
5,Transformed train set shape,"(8258, 15)"
6,Transformed test set shape,"(435, 15)"
7,Numeric features,15
8,Categorical features,1
9,Preprocess,True


Let's play around with top 3 models

In [16]:
top_models = compare_models(n_select = 3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8069,0.8985,0.8129,0.8055,0.8091,0.6137,0.6139,0.5620
catboost,CatBoost Classifier,0.8048,0.8991,0.8115,0.8030,0.8071,0.6095,0.6097,4.9280
gbc,Gradient Boosting Classifier,0.8016,0.8896,0.8355,0.7846,0.8091,0.6031,0.6047,0.7860
xgboost,Extreme Gradient Boosting,0.8014,0.8916,0.7963,0.8068,0.8015,0.6028,0.6029,1.0180
rf,Random Forest Classifier,0.7983,0.8823,0.7709,0.8180,0.7937,0.5967,0.5977,0.7500
et,Extra Trees Classifier,0.7957,0.8722,0.7540,0.8253,0.7880,0.5917,0.5940,0.6160
ada,Ada Boost Classifier,0.7928,0.8788,0.8230,0.7784,0.8000,0.5854,0.5865,0.4220
lr,Logistic Regression,0.7869,0.8690,0.8127,0.7751,0.7934,0.5736,0.5743,1.6060
knn,K Neighbors Classifier,0.7729,0.8501,0.7685,0.7781,0.7732,0.5459,0.5460,0.3720
ridge,Ridge Classifier,0.7625,0.0000,0.6997,0.8036,0.7480,0.5255,0.5301,0.1680


Processing:   0%|          | 0/71 [00:00<?, ?it/s]

In [17]:
top_models

[LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
                importance_type='split', learning_rate=0.1, max_depth=-1,
                min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
                n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
                random_state=8278, reg_alpha=0.0, reg_lambda=0.0, silent='warn',
                subsample=1.0, subsample_for_bin=200000, subsample_freq=0),
 GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                            learning_rate=0.1, loss='deviance', max_depth=3,
                            max_features=None, max_leaf_nodes=None,
                            min_impurity_decrease=0.0, min_samples_leaf=1,
                            min_samples_split=2, min_weight_fraction_leaf=0.0,
                            n_estimators=100, n_iter_no_change=None,
                            random_state=8278, subsample=1.0, tol=0.0001,
          

In [18]:
lgbm = create_model('lightgbm')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8123,0.9051,0.8221,0.8085,0.8153,0.6246,0.6247
1,0.8136,0.8987,0.8281,0.8068,0.8173,0.6270,0.6272
2,0.8063,0.9034,0.8293,0.7949,0.8118,0.6124,0.6130
3,0.8086,0.8970,0.8014,0.8152,0.8083,0.6172,0.6173
4,0.7935,0.8881,0.7837,0.8020,0.7927,0.5870,0.5871
Mean,0.8069,0.8985,0.8129,0.8055,0.8091,0.6137,0.6139
Std,0.0072,0.0060,0.0177,0.0068,0.0087,0.0143,0.0143


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
lgbm_tunned = tune_model(lgbm)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8148,0.9026,0.8197,0.8138,0.8168,0.6295,0.6295
1,0.8069,0.8953,0.8161,0.8036,0.8098,0.6137,0.6138
2,0.8093,0.9032,0.8113,0.8103,0.8108,0.6186,0.6186
3,0.8031,0.8952,0.7894,0.8139,0.8015,0.6064,0.6066
4,0.7880,0.8846,0.7704,0.8012,0.7855,0.5761,0.5766
Mean,0.8044,0.8962,0.8014,0.8086,0.8049,0.6089,0.6090
Std,0.0090,0.0067,0.0187,0.0053,0.0108,0.0180,0.0179


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [20]:
catboost = create_model('catboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8087,0.9033,0.8269,0.8000,0.8132,0.6173,0.6177
1,0.8099,0.9005,0.8245,0.8033,0.8138,0.6198,0.6200
2,0.8099,0.9042,0.8221,0.8047,0.8133,0.6198,0.6199
3,0.8019,0.8969,0.8002,0.8051,0.8027,0.6039,0.6039
4,0.7935,0.8906,0.7837,0.8020,0.7927,0.5870,0.5871
Mean,0.8048,0.8991,0.8115,0.8030,0.8071,0.6095,0.6097
Std,0.0064,0.0049,0.0169,0.0019,0.0083,0.0127,0.0128


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [21]:
catboost_tunned = tune_model(catboost)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8105,0.9045,0.8293,0.8014,0.8151,0.6209,0.6213
1,0.8136,0.9023,0.8353,0.8025,0.8186,0.6270,0.6275
2,0.8093,0.9026,0.8293,0.7995,0.8142,0.6185,0.6190
3,0.8068,0.8965,0.8051,0.8099,0.8075,0.6136,0.6136
4,0.8007,0.8911,0.7933,0.8078,0.8005,0.6015,0.6016
Mean,0.8082,0.8994,0.8185,0.8042,0.8112,0.6163,0.6166
Std,0.0043,0.0050,0.0163,0.0040,0.0064,0.0086,0.0087


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [22]:
gbc = create_model('gbc')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8087,0.8975,0.8498,0.7873,0.8173,0.6172,0.6192
1,0.8021,0.8887,0.8365,0.7847,0.8098,0.6039,0.6052
2,0.8063,0.8912,0.8606,0.7783,0.8174,0.6123,0.6158
3,0.8031,0.8885,0.8315,0.7888,0.8096,0.6061,0.6070
4,0.7880,0.8822,0.7993,0.7842,0.7917,0.5759,0.5760
Mean,0.8016,0.8896,0.8355,0.7846,0.8091,0.6031,0.6047
Std,0.0072,0.0050,0.0208,0.0036,0.0094,0.0144,0.0152


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [23]:
gbc_tunned = tune_model(gbc)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8099,0.8996,0.8185,0.8069,0.8126,0.6198,0.6199
1,0.8063,0.8959,0.8173,0.8019,0.8095,0.6125,0.6126
2,0.8105,0.9017,0.8269,0.8028,0.8147,0.6210,0.6212
3,0.8001,0.8955,0.7978,0.8036,0.8007,0.6002,0.6003
4,0.7947,0.8878,0.7849,0.8032,0.7939,0.5894,0.5895
Mean,0.8043,0.8961,0.8091,0.8037,0.8063,0.6086,0.6087
Std,0.0061,0.0048,0.0154,0.0017,0.0078,0.0121,0.0121


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits


### 6.1. Blending the top models

In [24]:
blend_model = blend_models(estimator_list = [lgbm_tunned, catboost_tunned, gbc_tunned])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8136,0.9057,0.8281,0.8068,0.8173,0.6270,0.6272
1,0.8081,0.9015,0.8221,0.8019,0.8119,0.6161,0.6163
2,0.8099,0.9050,0.8245,0.8033,0.8138,0.6198,0.6200
3,0.8110,0.8991,0.8063,0.8161,0.8111,0.6221,0.6221
4,0.7947,0.8910,0.7825,0.8047,0.7934,0.5894,0.5896
Mean,0.8075,0.9005,0.8127,0.8065,0.8095,0.6149,0.6151
Std,0.0066,0.0053,0.0169,0.0050,0.0083,0.0132,0.0132


Processing:   0%|          | 0/6 [00:00<?, ?it/s]

In [25]:
blend_model_tunned = tune_model(blend_model)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8136,0.9057,0.8281,0.8068,0.8173,0.6270,0.6272
1,0.8081,0.9015,0.8221,0.8019,0.8119,0.6161,0.6163
2,0.8099,0.9050,0.8245,0.8033,0.8138,0.6198,0.6200
3,0.8110,0.8991,0.8063,0.8161,0.8111,0.6221,0.6221
4,0.7947,0.8910,0.7825,0.8047,0.7934,0.5894,0.5896
Mean,0.8075,0.9005,0.8127,0.8065,0.8095,0.6149,0.6151
Std,0.0066,0.0053,0.0169,0.0050,0.0083,0.0132,0.0132


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [26]:
stacked_model = stack_models(estimator_list = [lgbm_tunned, catboost_tunned, gbc_tunned])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8136,0.9064,0.8377,0.8011,0.8190,0.6270,0.6276
1,0.8130,0.9022,0.8425,0.7975,0.8194,0.6257,0.6267
2,0.8081,0.9040,0.8389,0.7923,0.8149,0.6160,0.6171
3,0.8086,0.8988,0.8195,0.8040,0.8117,0.6171,0.6173
4,0.7959,0.8905,0.8077,0.7915,0.7995,0.5917,0.5918
Mean,0.8078,0.9004,0.8293,0.7973,0.8129,0.6155,0.6161
Std,0.0064,0.0055,0.0134,0.0049,0.0073,0.0127,0.0130


Processing:   0%|          | 0/6 [00:00<?, ?it/s]

[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.4, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 

In [27]:
final_model = finalize_model(stacked_model)
final_model

Pipeline(memory=Memory(location=/tmp/joblib),
         steps=[('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['HomePlanet', 'CryoSleep', 'Cabin',
                                             'Destination', 'Age', 'VIP',
                                             'RoomService', 'FoodCourt',
                                             'ShoppingMall', 'Spa', 'VRDeck',
                                             'Deck', 'Num', 'Side',
                                             'Relative_group'],
                                    transformer=SimpleImputer(add_indicator=False,
                                                              copy=True,
                                                              fill_value=None,
                                                              missing_val...
                                                                            verbose=0,
                             

In [28]:
predictions = predict_model(final_model, data = test_copy)
predictions.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Num,Side,Relative_group,Label,Score
0,0.0,1.0,0.622520,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000,0.036384,0.0,0.0,1,0.5233
1,0.0,0.0,0.133870,0.0,-0.470588,0.0,0.000000,0.147541,0.000000,53.264151,0.000,0.158765,0.0,0.0,0,0.9677
2,1.0,1.0,-0.707596,-2.0,0.235294,0.0,0.000000,0.000000,0.000000,0.000000,0.000,-1.195149,0.0,0.0,1,0.9562
3,1.0,0.0,-0.707239,0.0,0.647059,0.0,0.000000,109.049180,0.000000,3.415094,14.625,-1.194046,0.0,0.0,1,0.9642
4,0.0,0.0,0.173369,0.0,-0.411765,0.0,0.243902,0.000000,28.863636,0.000000,0.000,0.281147,0.0,0.0,1,0.5499


In [29]:
sub_sample.head()

,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,False
3,0021_01,False
4,0023_01,False


In [30]:
sub_sample.Transported = predictions['Label']
sub_sample['Transported'] = sub_sample['Transported'].astype('bool')
sub_sample.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [31]:
sub_sample.to_csv('submission.csv', index = False)